# MotionIQ — Phase 3: causal phase classifier (Colab GPU)

Run on a **GPU runtime** (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**.
Before running, drag `artifacts/train_data.npz` (105.9 MB) into the Files panel (left sidebar, folder icon) so it lands at `/content/train_data.npz`. If it isn't there, Cell 1 opens an upload dialog.

| Cell | What it does |
|---|---|
| 1 | install `onnx`/`onnxruntime`, load the bundle, assert keys/shapes/no-NaN |
| 2 | splits (internal subject-wise val carved from train subjects), causal windowing, model, train/eval code |
| 3 | ablation: 5 feature variants × 3 seeds, early stopping on **val** macro-F1; held-out evaluated once per run |
| 4 | export `model_geom.onnx` + `model_best.onnx`, ORT parity check, `metrics.json`, zip + download |
| 5 | (optional) TensorRT FP16 latency, reporting only; updates `metrics.json` and re-downloads the zip |

Model selection never looks at the held-out subjects. The model is causal: unidirectional LSTM over the last W frames, predicting the phase of the **last** frame (what the live demo sees).

In [ ]:
# ---- Cell 1: setup, upload, bundle assertions ----
import os, sys, json, time, copy, zipfile, subprocess
import numpy as np
import torch
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("MOTIONIQ_SMOKE") == "1"   # local Mac code check only: tiny slice, 1 epoch
DATA = os.environ.get("MOTIONIQ_DATA", "/content/train_data.npz")
OUT = os.environ.get("MOTIONIQ_OUT", "/content/motioniq_out")
os.makedirs(OUT, exist_ok=True)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
    if not os.path.exists(DATA):
        from google.colab import files
        up = files.upload()                      # pick artifacts/train_data.npz
        os.replace(next(iter(up)), DATA)
import onnx, onnxruntime as ort

d = dict(np.load(DATA))
N, C = len(d["phase"]), len(d["clip_id"])
EXPECT = {"geom": (N, 19), "valid": (N,), "fill": (N,), "kpts": (N, 17, 3), "dino": (N, 384), "resnet": (N, 512),
          "phase": (N,), "frame_clip": (N,), "frame_idx": (N,), "t": (N,), "frame_rep": (N,), "frame_split": (N,),
          "frame_faults": (N, 5), "clip_id": (C,), "subject_id": (C,), "view": (C,), "quality": (C,), "split": (C,),
          "clip_start": (C,), "clip_len": (C,), "geom_names": (19,), "phase_names": (4,), "fault_names": (5,),
          "split_names": (2,)}
for k, shp in EXPECT.items():
    assert k in d, f"missing key {k}"
    assert d[k].shape == shp, f"{k}: shape {d[k].shape} != {shp}"
assert (N, C) == (60123, 136), f"unexpected bundle size {(N, C)} (expected the Phase 2 bundle: 60123 frames, 136 clips)"
for k in ("geom", "dino", "resnet"):
    assert np.isfinite(d[k].astype(np.float32)).all(), f"NaN/inf in {k}"
assert list(d["phase_names"]) == ["standing", "descent", "bottom", "ascent"]
assert set(d["split"]) == {"train", "heldout"}
assert not set(d["subject_id"][d["split"] == "train"]) & set(d["subject_id"][d["split"] == "heldout"])
DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"bundle OK: {N} frames, {C} clips | torch {torch.__version__} | onnx {onnx.__version__} | ort {ort.__version__} | device {DEV}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEV == "cuda" else "") + (" | SMOKE TEST" if SMOKE else ""))

In [ ]:
# ---- Cell 2: splits, causal windows, model, train/eval ----
import torch.nn as nn
W = 32               # window = 32 frames = 2.13 s at 15 fps (~one squat rep); phase of the LAST frame
HIDDEN, LAYERS, DROPOUT = 128, 2, 0.2
BATCH, LR, WD = 256, 1e-3, 1e-2
EPOCHS, PATIENCE = (1, 1) if SMOKE else (40, 6)
SEEDS = [0] if SMOKE else [0, 1, 2]
N_VAL_SUBJ, VAL_SEED = 4, 0
PHASES = list(d["phase_names"])

# --- subject-wise roles: train / val (carved from the 20 train subjects) / test (the 5 held-out subjects)
subj, split = d["subject_id"], d["split"]
train_subj = sorted(map(str, set(subj[split == "train"])))
val_subj = sorted(np.random.RandomState(VAL_SEED).choice(train_subj, N_VAL_SUBJ, replace=False).tolist())
test_subj = sorted(map(str, set(subj[split == "heldout"])))
clip_role = np.where(split == "heldout", "test", np.where(np.isin(subj, val_subj), "val", "train"))
if SMOKE:  # tiny slice: 2 clips per role
    keep = np.concatenate([np.flatnonzero(clip_role == r)[:2] for r in ("train", "val", "test")])
    clip_role = np.where(np.isin(np.arange(C), keep), clip_role, "unused")
fclip = d["frame_clip"].astype(np.int64)
frame_role = clip_role[fclip]
IDX = {r: np.flatnonzero(frame_role == r) for r in ("train", "val", "test")}
fstart = d["clip_start"][fclip].astype(np.int64)      # first frame of each frame's clip (window clamp)
y_all = d["phase"].astype(np.int64)
print("subjects  train:", len(set(subj[clip_role == "train"])), " val:", val_subj, " test:", test_subj)
print("frames   ", {r: len(v) for r, v in IDX.items()})
assert not (set(subj[clip_role == "val"]) | set(subj[clip_role == "train"])) & set(test_subj)
assert not set(subj[clip_role == "val"]) & set(subj[clip_role == "train"])

# --- feature blocks and variants. Normalisation stats from TRAIN-role frames only; baked into the model.
BLOCKS = {"geom": d["geom"].astype(np.float32), "valid": d["valid"].astype(np.float32)[:, None],
          "resnet": d["resnet"].astype(np.float32), "dino": d["dino"].astype(np.float32)}
NAMES = {"geom": [str(n) for n in d["geom_names"]], "valid": ["valid"],
         "resnet": [f"resnet_{i}" for i in range(512)], "dino": [f"dino_cls_{i}" for i in range(384)]}
STATS = {}
for b, X in BLOCKS.items():
    if b == "valid":   # binary flag: passed through un-normalised
        STATS[b] = (np.zeros(1, np.float32), np.ones(1, np.float32))
    else:
        tr = X[IDX["train"]]
        STATS[b] = (tr.mean(0), tr.std(0) + 1e-6)
# geometry variants carry the valid flag; visual-only variants get no pose-derived input at all
VARIANTS = {"geom": ["geom", "valid"], "resnet": ["resnet"], "dino": ["dino"],
            "geom+resnet": ["geom", "valid", "resnet"], "geom+dino": ["geom", "valid", "dino"]}

def variant_arrays(v):
    X = np.concatenate([BLOCKS[b] for b in VARIANTS[v]], 1)
    mu = np.concatenate([STATS[b][0] for b in VARIANTS[v]])
    sd = np.concatenate([STATS[b][1] for b in VARIANTS[v]])
    return X, mu, sd

class CausalPhaseLSTM(nn.Module):
    """(B, W, D) raw features, oldest..newest -> (B, 4) logits for the newest frame. Unidirectional = causal."""
    def __init__(self, D, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32))
        self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32))
        self.inp = nn.Sequential(nn.Dropout(0.1), nn.Linear(D, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(), nn.Dropout(DROPOUT))
        self.lstm = nn.LSTM(HIDDEN, HIDDEN, num_layers=LAYERS, batch_first=True, dropout=DROPOUT)
        self.head = nn.Linear(HIDDEN, len(PHASES))
    def forward(self, x):
        h, _ = self.lstm(self.inp((x - self.mean) / self.std))
        return self.head(h[:, -1])

OFFS = torch.arange(-(W - 1), 1)
def windows(Xt, idx):
    """Frames idx-W+1..idx of the same clip; before the clip start, the first frame is repeated
    (the live demo pads the same way until it has W frames)."""
    i = torch.as_tensor(idx)
    g = torch.maximum(i[:, None] + OFFS, torch.as_tensor(fstart[idx])[:, None])
    return Xt[g.to(Xt.device)]

def confusion(y, p, k=4):
    return np.bincount(y * k + p, minlength=k * k).reshape(k, k)

def scores(y, p):
    cm = confusion(y, p)
    tp = np.diag(cm).astype(float)
    f1 = 2 * tp / np.maximum(cm.sum(0) + cm.sum(1), 1)
    return {"acc": float(tp.sum() / cm.sum()), "macro_f1": float(f1.mean()),
            "f1_per_phase": dict(zip(PHASES, f1.round(4).tolist())), "confusion": cm.tolist()}

@torch.no_grad()
def predict(model, Xt, idx, bs=4096):
    model.eval()
    return np.concatenate([model(windows(Xt, idx[s:s + bs])).argmax(1).cpu().numpy() for s in range(0, len(idx), bs)])

def train_run(v, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    X, mu, sd = variant_arrays(v)
    Xt = torch.from_numpy(X).to(DEV)
    model = CausalPhaseLSTM(X.shape[1], mu, sd).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    cnt = np.bincount(y_all[IDX["train"]], minlength=4).astype(np.float32)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum() / (4 * np.maximum(cnt, 1))).to(DEV))
    yt = torch.from_numpy(y_all).to(DEV)
    best, best_state, best_ep, bad, t0 = -1.0, None, -1, 0, time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = IDX["train"][np.random.permutation(len(IDX["train"]))]
        for s in range(0, len(perm), BATCH):
            b = perm[s:s + BATCH]
            loss = lossf(model(windows(Xt, b)), yt[torch.as_tensor(b).to(DEV)])
            opt.zero_grad(); loss.backward(); opt.step()
        vf1 = scores(y_all[IDX["val"]], predict(model, Xt, IDX["val"]))["macro_f1"]
        if vf1 > best:
            best, best_state, best_ep, bad = vf1, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    res = {"variant": v, "seed": seed, "dim": X.shape[1], "best_epoch": best_ep, "epochs_run": ep + 1,
           "train_time_s": round(time.time() - t0, 1),
           "val": scores(y_all[IDX["val"]], predict(model, Xt, IDX["val"])),
           "test": scores(y_all[IDX["test"]], predict(model, Xt, IDX["test"]))}   # held-out: reported, never selected on
    res["state"] = {k: t.detach().cpu() for k, t in model.state_dict().items()}
    del Xt
    return res
print("variants:", {v: variant_arrays(v)[0].shape[1] for v in VARIANTS}, "| W =", W)

In [ ]:
# ---- Cell 3: ablation (same model / splits / hyper-parameters), 3 seeds per variant ----
RUNS = []
for v in VARIANTS:
    for seed in SEEDS:
        r = train_run(v, seed)
        RUNS.append(r)
        print(f"{v:12s} seed {seed}  best ep {r['best_epoch']:2d}/{r['epochs_run']:2d}  {r['train_time_s']:6.1f}s  "
              f"val acc {r['val']['acc']:.3f} F1 {r['val']['macro_f1']:.3f} | test acc {r['test']['acc']:.3f} F1 {r['test']['macro_f1']:.3f}")

def agg(v):
    rs = [r for r in RUNS if r["variant"] == v]
    ms = lambda part, key: (float(np.mean([r[part][key] for r in rs])), float(np.std([r[part][key] for r in rs])))
    return {"dim": rs[0]["dim"], "seeds": [r["seed"] for r in rs],
            **{f"{part}_{key}_mean_std": ms(part, key) for part in ("val", "test") for key in ("acc", "macro_f1")},
            "test_f1_per_phase_mean": {p: float(np.mean([r["test"]["f1_per_phase"][p] for r in rs])) for p in PHASES},
            "test_confusion_sum_over_seeds": np.sum([r["test"]["confusion"] for r in rs], 0).tolist(),
            "runs": [{k: r[k] for k in ("seed", "best_epoch", "epochs_run", "train_time_s", "val", "test")} for r in rs]}
ABL = {v: agg(v) for v in VARIANTS}
print(f"\n{'variant':12s} {'dim':>5s}   {'val macro-F1':>14s}   {'TEST acc':>14s}   {'TEST macro-F1':>14s}   test F1 per phase (mean)")
for v, a in ABL.items():
    f = lambda t: f"{t[0]:.3f}±{t[1]:.3f}"
    print(f"{v:12s} {a['dim']:5d}   {f(a['val_macro_f1_mean_std']):>14s}   {f(a['test_acc_mean_std']):>14s}   "
          f"{f(a['test_macro_f1_mean_std']):>14s}   " + " ".join(f"{p[:4]} {x:.3f}" for p, x in a["test_f1_per_phase_mean"].items()))
# selection uses VALIDATION only
BEST_VARIANT = max([v for v in VARIANTS if v != "geom"], key=lambda v: ABL[v]["val_macro_f1_mean_std"][0])
pick = lambda v: max([r for r in RUNS if r["variant"] == v], key=lambda r: r["val"]["macro_f1"])
EXPORT = {"model_geom.onnx": pick("geom"), "model_best.onnx": pick(BEST_VARIANT)}
print("\nbest non-geometry variant by val macro-F1:", BEST_VARIANT,
      "| exported seeds:", {k: (r["variant"], r["seed"]) for k, r in EXPORT.items()})
print("test confusion (rows=true, cols=pred, summed over seeds) for", BEST_VARIANT, ":\n",
      np.array(ABL[BEST_VARIANT]["test_confusion_sum_over_seeds"]))

In [ ]:
# ---- Cell 4: ONNX export + parity check + metrics.json + zip ----
PARITY, LAT = {}, {}
rng = np.random.default_rng(0)
for fname, r in EXPORT.items():
    v = r["variant"]
    X, mu, sd = variant_arrays(v)
    m = CausalPhaseLSTM(X.shape[1], mu, sd)
    m.load_state_dict(r["state"]); m.eval()
    path = os.path.join(OUT, fname)
    dummy = torch.zeros(1, W, X.shape[1])
    kw = dict(input_names=["features"], output_names=["logits"], opset_version=17)
    try:
        torch.onnx.export(m, dummy, path, dynamo=False, **kw)
    except TypeError:          # older torch without the dynamo flag
        torch.onnx.export(m, dummy, path, **kw)
    onnx.checker.check_model(onnx.load(path))
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    Xt = torch.from_numpy(X)
    sample = rng.choice(IDX["test"], 64, replace=False)
    diffs, agree = [], 0
    with torch.no_grad():
        for i in sample:
            x = windows(Xt, np.array([i]))                  # fixed input shape (1, W, D)
            a, b = m(x).numpy(), sess.run(None, {"features": x.numpy()})[0]
            diffs.append(float(np.abs(a - b).max())); agree += int(a.argmax() == b.argmax())
    PARITY[fname] = {"variant": v, "max_abs_diff": max(diffs), "argmax_agreement": agree / len(sample), "n_windows": len(sample)}
    x = windows(Xt, sample[:1]).numpy()
    for _ in range(10): sess.run(None, {"features": x})
    ts = []
    for _ in range(200):
        t0 = time.perf_counter(); sess.run(None, {"features": x}); ts.append((time.perf_counter() - t0) * 1e3)
    LAT[fname] = {"onnxruntime_cpu_ms_median": float(np.median(ts)), "onnxruntime_cpu_ms_p95": float(np.percentile(ts, 95))}
    if DEV == "cuda":
        mg, xg = m.to("cuda"), torch.from_numpy(x).cuda()
        with torch.no_grad():
            for _ in range(20): mg(xg)
            torch.cuda.synchronize(); t0 = time.perf_counter()
            for _ in range(200): mg(xg)
            torch.cuda.synchronize()
        LAT[fname]["pytorch_gpu_fp32_ms_mean"] = (time.perf_counter() - t0) / 200 * 1e3
    assert PARITY[fname]["max_abs_diff"] < 1e-4, PARITY[fname]
    print(fname, PARITY[fname], LAT[fname])

def feat_order(v):
    return {"blocks": VARIANTS[v], "dims": {b: len(NAMES[b]) for b in VARIANTS[v]},
            "total_dim": int(sum(len(NAMES[b]) for b in VARIANTS[v])),
            "geom_names": NAMES["geom"] if "geom" in VARIANTS[v] else None}
METRICS = {
    "task": "per-frame squat phase, causal (predicts the last frame of a W-frame window)",
    "label_mapping": {i: p for i, p in enumerate(PHASES)},
    "causal": True, "W": W, "fps": int(d["fps"]),
    "window_padding": "frames before the clip start are filled with the clip's first frame (live demo: repeat the first frame until W frames exist)",
    "onnx_io": {"input": "features float32 (1, W, D) raw (un-normalised) features, oldest..newest",
                "output": "logits float32 (1, 4)", "normalisation": "inside the model ((x - mean) / std buffers)"},
    "model": {"type": "unidirectional LSTM", "hidden": HIDDEN, "layers": LAYERS, "dropout": DROPOUT,
              "input_proj": "Dropout(0.1) -> Linear(D,128) -> LayerNorm -> ReLU -> Dropout"},
    "training": {"optimizer": "AdamW", "lr": LR, "weight_decay": WD, "batch": BATCH, "max_epochs": EPOCHS,
                 "early_stopping_patience": PATIENCE, "selection": "val macro-F1", "loss": "class-weighted cross-entropy",
                 "seeds": SEEDS, "smoke_test": SMOKE},
    "splits": {"train_subjects": sorted(map(str, set(subj[clip_role == "train"]))), "val_subjects": val_subj,
               "test_subjects(heldout)": test_subj, "val_seed": VAL_SEED,
               "frames": {r: int(len(v)) for r, v in IDX.items()},
               "note": "val carved subject-wise from the 20 train subjects; held-out used only for reported test numbers"},
    "variants": {v: feat_order(v) for v in VARIANTS},
    "normalisation_train_only": {b: {"mean": STATS[b][0].round(6).tolist(), "std": STATS[b][1].round(6).tolist()} for b in STATS},
    "ablation": ABL, "best_variant_by_val_macro_f1": BEST_VARIANT,
    "exported": {f: {"variant": r["variant"], "seed": r["seed"], "val_macro_f1": r["val"]["macro_f1"],
                     "test_acc": r["test"]["acc"], "test_macro_f1": r["test"]["macro_f1"], **feat_order(r["variant"])}
                 for f, r in EXPORT.items()},
    "onnx_parity": PARITY, "latency": LAT,
    "env": {"torch": torch.__version__, "onnx": onnx.__version__, "onnxruntime": ort.__version__,
            "device": DEV, "gpu": torch.cuda.get_device_name(0) if DEV == "cuda" else None},
}
def write_and_zip():
    with open(os.path.join(OUT, "metrics.json"), "w") as fh:
        json.dump(METRICS, fh, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    zp = os.path.join(OUT, "motioniq_artifacts.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in ("model_geom.onnx", "model_best.onnx", "metrics.json"):
            z.write(os.path.join(OUT, f), f)
    print("wrote", zp, f"({os.path.getsize(zp) / 1e6:.2f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(zp)
    return zp
write_and_zip()

In [ ]:
# ---- Cell 5 (optional): TensorRT FP16 latency — reporting only, engine never leaves Colab ----
if DEV != "cuda":
    print("no CUDA GPU: TensorRT benchmark skipped")
else:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorrt"], check=True)
        import tensorrt as trt
        logger = trt.Logger(trt.Logger.WARNING)
        for fname in EXPORT:
            builder = trt.Builder(logger)
            flag = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH) if hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH") else 0
            net = builder.create_network(flag)
            parser = trt.OnnxParser(net, logger)
            if not parser.parse(open(os.path.join(OUT, fname), "rb").read()):
                raise RuntimeError("; ".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
            cfg = builder.create_builder_config()
            cfg.set_flag(trt.BuilderFlag.FP16)
            eng = trt.Runtime(logger).deserialize_cuda_engine(builder.build_serialized_network(net, cfg))
            ctx = eng.create_execution_context()
            D = METRICS["exported"][fname]["total_dim"]
            xin = torch.randn(1, W, D, device="cuda"); yout = torch.empty(1, 4, device="cuda")
            ctx.set_tensor_address("features", xin.data_ptr()); ctx.set_tensor_address("logits", yout.data_ptr())
            st = torch.cuda.current_stream()
            for _ in range(50): ctx.execute_async_v3(st.cuda_stream)
            torch.cuda.synchronize()
            e0, e1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            e0.record()
            for _ in range(500): ctx.execute_async_v3(st.cuda_stream)
            e1.record(); torch.cuda.synchronize()
            LAT[fname]["tensorrt_fp16_ms_mean"] = e0.elapsed_time(e1) / 500
            LAT[fname]["tensorrt_version"] = trt.__version__
            print(fname, LAT[fname])
        METRICS["latency"] = LAT
        write_and_zip()   # downloads the zip again with the TensorRT numbers
    except Exception as e:
        print("TensorRT benchmark failed (optional, reporting only; the zip from Cell 4 is complete):", repr(e))